# Aula 17 - Problemas Hamiltonianos e Roteamento Logístico de AGVs (TSP)

Implementação heurística (Nearest Neighbor + 2-Opt) para definir a menor trajetória para calibração de sensores na Estação H2.

In [1]:
from typing import List

class RoteadorCalibracao_TSP:
    def __init__(self, locais: List[str], matriz_distancias: List[List[float]]):
        self.locais = locais
        self.dist = matriz_distancias
        self.n = len(locais)
        
    def calcular_custo_rota(self, rota: List[int]) -> float:
        c = sum(self.dist[rota[i]][rota[i+1]] for i in range(len(rota)-1))
        c += self.dist[rota[-1]][rota[0]]
        return c

    def vizinho_mais_proximo(self, inicio_idx: int = 0) -> List[int]:
        nao_vis = set(range(self.n))
        nao_vis.remove(inicio_idx)
        rota = [inicio_idx]
        atual = inicio_idx
        while nao_vis:
            prox = min(nao_vis, key=lambda v: self.dist[atual][v])
            rota.append(prox)
            nao_vis.remove(prox)
            atual = prox
        return rota

    def otimizar_2opt(self, rota: List[int]) -> List[int]:
        melhor = list(rota)
        melhor_c = self.calcular_custo_rota(melhor)
        melhorou = True
        while melhorou:
            melhorou = False
            for i in range(1, self.n - 1):
                for j in range(i + 1, self.n):
                    nova = melhor[:i] + melhor[i:j+1][::-1] + melhor[j+1:]
                    c = self.calcular_custo_rota(nova)
                    if c < melhor_c:
                        melhor_c = c
                        melhor = nova
                        melhorou = True
                        break
                if melhorou: break
        return melhor

locais = ["Base/Sala de Controle", "E-101", "C-101", "C-102", "Bateria TK-101", "Bateria TK-102", "MAN-101", "D-101"]
dist_mat = [
    [0.0, 10.0, 20.0, 25.0, 40.0, 45.0, 35.0, 15.0],
    [10.0, 0.0, 15.0, 30.0, 45.0, 50.0, 40.0, 20.0],
    [20.0, 15.0, 0.0, 10.0, 25.0, 30.0, 25.0, 35.0],
    [25.0, 30.0, 10.0, 0.0, 15.0, 20.0, 18.0, 40.0],
    [40.0, 45.0, 25.0, 15.0, 0.0, 12.0, 20.0, 50.0],
    [45.0, 50.0, 30.0, 20.0, 12.0, 0.0, 15.0, 55.0],
    [35.0, 40.0, 25.0, 18.0, 20.0, 15.0, 0.0, 22.0],
    [15.0, 20.0, 35.0, 40.0, 50.0, 55.0, 22.0, 0.0]
]

agv_tsp = RoteadorCalibracao_TSP(locais, dist_mat)
r_ini = agv_tsp.vizinho_mais_proximo(0)
r_opt = agv_tsp.otimizar_2opt(r_ini)

print("--- PLANO LOGÍSTICO DE CALIBRAÇÃO DE SENSORES H2 ---")
print("Rota Inicial NN:", " -> ".join([locais[i] for i in r_ini]) + f"\nCusto Total (Metros): {agv_tsp.calcular_custo_rota(r_ini):.1f}m\n")
print("Rota Ótima 2-Opt (Pós-Refinamento):", " -> ".join([locais[i] for i in r_opt]) + f"\nCusto Total (Metros): {agv_tsp.calcular_custo_rota(r_opt):.1f}m")


--- PLANO LOGÍSTICO DE CALIBRAÇÃO DE SENSORES H2 ---
Rota Inicial NN: Base/Sala de Controle -> E-101 -> C-101 -> C-102 -> Bateria TK-101 -> Bateria TK-102 -> MAN-101 -> D-101
Custo Total (Metros): 150.0m

Rota Ótima 2-Opt (Pós-Refinamento): Base/Sala de Controle -> E-101 -> C-101 -> C-102 -> Bateria TK-101 -> Bateria TK-102 -> MAN-101 -> D-101
Custo Total (Metros): 150.0m
